In [ ]:
# =========================
# Step 1: Import Libraries
# =========================
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
import joblib

# =========================
# Step 2: Load Data
# =========================
train = pd.read_csv("../data/application_train.csv")
test = pd.read_csv("../data/application_test.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)

# =========================
# Step 3: Quick Data Cleaning
# =========================
# Drop columns with >40% missing values
threshold = 0.4
train = train.loc[:, train.isnull().mean() < threshold]
test = test.loc[:, test.isnull().mean() < threshold]

# Fill remaining missing numeric values with median
train = train.fillna(train.median(numeric_only=True))
test = test.fillna(test.median(numeric_only=True))

# =========================
# Step 4: Encode Categorical Variables
# =========================
train = pd.get_dummies(train)
test = pd.get_dummies(test)

# Separate features and target
X = train.drop("TARGET", axis=1)
y = train["TARGET"]

# Align train and test columns
X, test = X.align(test, join='left', axis=1, fill_value=0)

# =========================
# Step 5: Clean Column Names for LightGBM
# =========================
X.columns = X.columns.str.replace('[^0-9a-zA-Z_]+', '_', regex=True)
test.columns = test.columns.str.replace('[^0-9a-zA-Z_]+', '_', regex=True)

print("Features shape:", X.shape)

# =========================
# Step 6: Train-Test Split
# =========================
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# =========================
# Step 7: Train LightGBM Model (Safe for any version)
# =========================
lgb_train = lgb.Dataset(X_train, y_train)
lgb_val = lgb.Dataset(X_val, y_val, reference=lgb_train)

params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'verbose': -1,
    'seed': 42
}

# Use callbacks for early stopping (version-independent)
model = lgb.train(
    params,
    lgb_train,
    num_boost_round=500,
    valid_sets=[lgb_train, lgb_val],
    callbacks=[lgb.early_stopping(stopping_rounds=50), lgb.log_evaluation(50)]
)

# =========================
# Step 8: Evaluate Model
# =========================
val_preds = model.predict(X_val)
auc = roc_auc_score(y_val, val_preds)
print("Validation ROC-AUC:", auc)

# =========================
# Step 9: Save Model
# =========================
joblib.dump(model, "../model/loan_model.pkl")
print("Model saved to ../model/loan_model.pkl")

Train shape: (307511, 122)
Test shape: (48744, 121)
Features shape: (307511, 184)
Training until validation scores don't improve for 50 rounds
[50]	training's auc: 0.751343	valid_1's auc: 0.740124
[100]	training's auc: 0.7687	valid_1's auc: 0.74942
[150]	training's auc: 0.780636	valid_1's auc: 0.752002
[200]	training's auc: 0.790804	valid_1's auc: 0.752234
[250]	training's auc: 0.79999	valid_1's auc: 0.752434
[300]	training's auc: 0.808866	valid_1's auc: 0.752424
Early stopping, best iteration is:
[292]	training's auc: 0.807479	valid_1's auc: 0.752517
Validation ROC-AUC: 0.7525173182631459
Model saved to ../model/loan_model.pkl


Bad pipe message: %s [b'0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7\r\nHost: localhost:35547\r\nUs', b'-Agent: Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.']
Bad pipe message: %s [b'0.0 Safari/537.36\r\nAccept-Encoding: gzip, defla']
Bad pipe message: %s [b', br, zstd\r\nAccept-Language: en-US,en;q=0.9\r\nCache-Control: max-age=0\r\nReferer: https://github.com/\r\nX-Request-ID: ', b'b0f34acf90ef42248d7eb39b286d9d\r\nX-Real-IP: 136.233.9.107\r\nX-Forwarded-Port: 443\r\nX-Forwarded-Sche', b': https\r\nX-Original-URI: /\r\nX-Scheme: https\r\nsec-fetch-site: cross-site\r\nsec-fetch-mode: navigate\r\nsec-fetch', b'est: document\r\nsec-ch-ua: "Chromium";v="146"']
Bad pipe message: %s [b'"Not-A.Brand";v="24", "Google Chrome";v="14']
Bad pipe message: %s [b'\r\nsec-ch-ua-mobile: ?0\r\nsec-ch-ua-platform: "Windows"']
Bad pipe message: %s [b'priority: u=']
Bad pipe message: %s [b' i\r\nX-Forwarded-Proto: ht

In [7]:
import joblib

# Load the trained LightGBM model
model = joblib.load("../model/loan_model.pkl")

# Correct way to get feature names for a Booster
feature_names = model.feature_name()
print(feature_names[:20])  # first 20 features

['SK_ID_CURR', 'CNT_CHILDREN', 'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE', 'REGION_POPULATION_RELATIVE', 'DAYS_BIRTH', 'DAYS_EMPLOYED', 'DAYS_REGISTRATION', 'DAYS_ID_PUBLISH', 'FLAG_MOBIL', 'FLAG_EMP_PHONE', 'FLAG_WORK_PHONE', 'FLAG_CONT_MOBILE', 'FLAG_PHONE', 'FLAG_EMAIL', 'CNT_FAM_MEMBERS', 'REGION_RATING_CLIENT', 'REGION_RATING_CLIENT_W_CITY']


In [8]:
import joblib

# Compute mean of each feature in training data
feature_means = X_train.mean()

# Save the means to use in FastAPI
joblib.dump(feature_means, "../model/feature_means.pkl")

['../model/feature_means.pkl']